In [ ]:
!pip install exifread

# Extracting GPS Information from Images

(You will need to modify this script based on how your dataset is stored in order to execute the code.).


In [ ]:
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
import os

# train_or_test_or_validation could be either train, test, or validation.
train_or_test_or_validation = "" # it could also be test or validation
PATH_TO_YOUR_DATA_FOLDER = "/content/drive/Shareddrives/CIS5190_final_project/photo_database_jpg"
directory_path = f"{PATH_TO_YOUR_DATA_FOLDER}/{train_or_test_or_validation}"
output_csv = "metadata.csv"
output_csv = os.path.join(directory_path, output_csv)

In [ ]:
import exifread, csv

def get_exif_data(image_path):
    with open(image_path, 'rb') as image_file:
        tags = exifread.process_file(image_file)
    return tags

def export_exif_to_json(exif_data, output_file):
    # Convert tags to a serializable format
    exif_data_serializable = {str(tag): str(value) for tag, value in exif_data.items()}
    with open(output_file, 'w') as json_file:
        json.dump(exif_data_serializable, json_file, indent=4)

In [ ]:
# Function to convert GPS coordinates in degrees, minutes, and seconds to decimal degrees
def convert_to_decimal_degrees(value):
    d, m, s = value.values
    return d.num / d.den + (m.num / m.den) / 60 + (s.num / s.den) / 3600

### You will need to create subfolders in {PATH_TO_YOUR_DATA_FOLDER} for each split (train/test/validation) or just (train/test). Next, place the corresponding images into each split after randomly shuffling them. Then, create a metadata.csv file for each split and place it in the corresponding directory. Note that the current code only works for jpeg images. If the exported images are in some other format, convert them to .jpg before running this code.

In [ ]:
import os
import csv
from tqdm import tqdm

files = [
    f for f in os.listdir(directory_path)
    if os.path.isfile(os.path.join(directory_path, f))
]

count_total = 0
count_written = 0

with open(output_csv, mode='w', newline='') as csv_file:
    fieldnames = ['file_name', 'Latitude', 'Longitude']
    writer = csv.DictWriter(csv_file, fieldnames=fieldnames)
    writer.writeheader()

    for filename in tqdm(files):
        file_path = os.path.join(directory_path, filename)
        count_total += 1

        try:
            exif_data = get_exif_data(file_path)
            if exif_data:
                gps_latitude = exif_data.get('GPS GPSLatitude', None)
                gps_latitude_ref = exif_data.get('GPS GPSLatitudeRef', None)
                gps_longitude = exif_data.get('GPS GPSLongitude', None)
                gps_longitude_ref = exif_data.get('GPS GPSLongitudeRef', None)

                if gps_latitude and gps_latitude_ref and gps_longitude and gps_longitude_ref:
                    latitude = convert_to_decimal_degrees(gps_latitude)
                    longitude = convert_to_decimal_degrees(gps_longitude)

                    if gps_latitude_ref.values[0] == 'S':
                        latitude = -latitude
                    if gps_longitude_ref.values[0] == 'W':
                        longitude = -longitude

                    writer.writerow({
                        'file_name': filename,
                        'Latitude': latitude,
                        'Longitude': longitude
                    })
                    count_written += 1
        except Exception as e:
            print(f"skip {filename}: {e}")

print("total files checked:", count_total)
print("rows written:", count_written)
print("saved to:", output_csv)

# Uploading and Reading a Dataset on Hugging Face

In [ ]:
!pip install datasets

In [ ]:
from datasets import load_dataset

folder_path = "/content/drive/Shareddrives/CIS5190_final_project/processed_dataset_grouped"

dataset = load_dataset("imagefolder", data_dir=folder_path)
print(dataset)
print(dataset["train"].features)
print(dataset["train"][0])

In [ ]:
!pip -q install huggingface_hub datasets
from huggingface_hub import login

login()

In [ ]:
from huggingface_hub import create_repo

repo_id = "JX128/image2GPS_project_dataset_grouped"
create_repo(repo_id=repo_id, repo_type="dataset", exist_ok=True)

dataset.push_to_hub(repo_id)